# IEEE 33-Bus OPF 

Time-series OPF (branch flow model) for the IEEE 33-bus system using only the rated line and load data. Objective: minimize total active power losses.

## How to Run This Notebook

This notebook is self-contained: it does not need any other files.

### 1. Install the Python packages
Run in a terminal (or Anaconda Prompt):
```
pip install pyomo numpy pandas scipy matplotlib ipykernel
```

### 2. Install the IPOPT solver
Pyomo does **not** include the IPOPT solver; it must be installed separately.
The solver setup cell after the imports installs IPOPT automatically if it is missing. To install it yourself instead, use **one** of these options:

- **With Anaconda:**
  ```
  conda install -c conda-forge ipopt
  ```
- **Without conda:**
  ```
  pip install idaes-pse
  idaes get-extensions
  ```

### 3. Select the kernel
In Jupyter or VS Code, select the Python environment where you installed the packages and IPOPT in steps 1 and 2.

### 4. Run the cells
Run the cells **in order from top to bottom** (Shift+Enter), or use **Run All**. Each cell depends on the cells above it.
If you change a value in the **Configuration** cell (e.g. `N`, `Vmag`), re-run that cell and all cells below it.

### Outputs
Results are saved to `Results/Obj(MinLosses)/Results_<Vmag>pu/` in the same folder as the notebook:
- `Voltage_Profiles_Data.csv`: bus voltages (p.u.) for each hour
- `Pg_Data.pkl`, `Total_losses_Data.pkl`, `Branches_Data.pkl`: substation power, total losses, and branch flows
- `Voltage_Profiles.pdf`: voltage profile figure


## Imports

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import os
from scipy.interpolate import interp1d
from pyomo.environ import *

In [2]:
# Solver setup: installs IPOPT if it is not already available (one-time, about 1 minute)
import shutil, subprocess, sys
if not SolverFactory('ipopt').available(exception_flag=False):
    if shutil.which('conda'):
        subprocess.run(['conda', 'install', '-y', '-q', '-c', 'conda-forge', 'ipopt'], check=True)
    else:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'idaes-pse'], check=True)
        subprocess.run(['idaes', 'get-extensions'], check=True)
        import idaes  # puts the IDAES binary folder (with ipopt) on PATH
print('IPOPT available:', SolverFactory('ipopt').available(exception_flag=False))

IPOPT available: True


## Configuration

In [3]:
import numpy as np

# Base Values (IEEE 33-bus, Baran & Wu 1989)
V_base = 12.66e3  # line to line
S_base = 1e6
I_base = S_base / (V_base * np.sqrt(3))

# Substation (slack) voltage magnitude, p.u.
Vmag = 1.0

# Bus voltage limits, p.u.
V_min = 0.90
V_max = 1.05

# Time series: number of time steps
days=1
N = days*24

## Network Data

In [4]:
# This file stores the line and load data for the IEEE 33-bus system 

# time index
t = np.arange(0, N, 1)

# Line flow variables 
flow_vars =[ (0,1),
             (1,2),
             (2,3),
             (3,4),
             (4,5),
             (5,6),
             (6,7),
             (7,8),
             (8,9),
             (9,10),
             (10,11),
             (11,12),
             (12,13),
             (13,14),
             (14,15),
             (15,16),
             (16,17),
             (1,18),
             (18,19),
             (19,20), 
             (20,21),
             (2,22),
             (22,23),
             (23,24), 
             (5,25),
             (25,26),
             (26,27),
             (27,28),
             (28,29),
             (29,30),
             (30,31),
             (31,32)]


# line impedances
Z_base = (V_base ** 2) / S_base


Z_Nominal = {key: val for key, val in {
            (0,1)  : 0.0922+ 0.0470j,
            (1,2)  : 0.4930+ 0.2511j,
            (2,3)  : 0.3660+ 0.1864j,
            (3,4)  : 0.3811+ 0.1941j,
            (4,5)  : 0.8190+ 0.7070j,
            (5,6)  : 0.1872+ 0.6188j,
            (6,7)  : 0.7114+ 0.2351j,
            (7,8)  : 1.0300+ 0.7400j,
            (8,9)  : 1.0440+ 0.7400j,
            (9,10) : 0.1966+ 0.0650j,
            (10,11): 0.3744+ 0.1238j,
            (11,12): 1.4680+ 1.1550j,
            (12,13): 0.5416+ 0.7129j,
            (13,14): 0.5910+ 0.5260j,
            (14,15): 0.7463+ 0.5450j,
            (15,16): 1.2890+ 1.7210j,
            (16,17): 0.7320+ 0.5740j,
            (1,18) : 0.1640+ 0.1565j,
            (18,19): 1.5042+ 1.3554j,
            (19,20): 0.4095+ 0.4784j,
            (20,21): 0.7089+ 0.9373j,
            (2,22) : 0.4512+ 0.3083j,
            (22,23): 0.8980+ 0.7091j,
            (23,24): 0.8960+ 0.7011j,
            (5,25) : 0.2030+ 0.1034j,
            (25,26): 0.2842+ 0.1447j,
            (26,27): 1.0590+ 0.9337j,
            (27,28): 0.8042+ 0.7006j,
            (28,29): 0.5075+ 0.2585j,
            (29,30): 0.9744+ 0.9630j,
            (30,31): 0.3105+ 0.3619j,
            (31,32): 0.3410+ 0.5302j}.items()}

# Convert impedances to per unit
Z = {key: val / Z_base for key, val in Z_Nominal.items()}

# bus variables
bus_vars = list(range(0,33))

# net injection to power flow mapping
bus2flow = {0  : [(0,1)],
            1  : [(0,1),(1,2),(1,18)],
            18 : [(1,18),(18,19)],
            19 : [(18,19),(19,20)],
            20 : [(19,20),(20,21)],
            21 : [(20,21)],
            2  : [(1,2),(2,3),(2,22)],
            22 : [(2,22),(22,23)],
            23 : [(22,23),(23,24)],
            24 : [(23,24)],
            3  : [(2,3),(3,4)],
            4  : [(3,4),(4,5)],
            5  : [(4,5),(5,6),(5,25)],
            25 : [(5,25),(25,26)],
            26 : [(25,26),(26,27)],
            27 : [(26,27),(27,28)],
            28 : [(27,28),(28,29)],
            29 : [(28,29),(29,30)],
            30 : [(29,30),(30,31)],
            31 : [(30,31),(31,32)],
            32 : [(31,32)],
            6  : [(5,6),(6,7)],
            7  : [(6,7),(7,8)],
            8  : [(7,8),(8,9)],
            9  : [(8,9),(9,10)],
            10 : [(9,10),(10,11)],
            11 : [(10,11),(11,12)],
            12 : [(11,12),(12,13)],
            13 : [(12,13),(13,14)],
            14 : [(13,14),(14,15)],
            15 : [(14,15),(15,16)],
            16 : [(15,16),(16,17)],
            17 : [(16,17)]}

# P/Q loads
load_p_nominal = { key: val for key, val in {
            0 : 0,
            1 : 100e3, 
            2 : 90e3,
            3 : 120e3, 
            4 : 60e3,
            5 : 60e3,
            6 : 200e3,
            7 : 200e3,
            8 : 60e3, 
            9 : 60e3,
            10: 45e3, 
            11: 60e3, 
            12: 60e3, 
            13: 120e3, 
            14: 60e3, 
            15: 60e3, 
            16: 60e3, 
            17: 90e3,
            18: 90e3, 
            19: 90e3,
            20: 90e3,
            21: 90e3, 
            22: 90e3, 
            23: 420e3,
            24: 420e3, 
            25: 60e3, 
            26: 60e3, 
            27: 60e3, 
            28: 120e3, 
            29: 200e3,
            30: 150e3, 
            31: 210e3,
            32: 60e3}.items()}

load_q_nominal = { key: val for key, val in {
            0 : 0,
            1 : 60e3,
            2 : 40e3,
            3 : 80e3,
            4 : 30e3,
            5 : 20e3,
            6 : 100e3,
            7 : 100e3,
            8 : 20e3,
            9 : 20e3,
            10: 30e3,
            11: 35e3,
            12: 35e3,
            13: 80e3,
            14: 10e3,
            15: 20e3,
            16: 20e3,
            17: 40e3,
            18: 40e3,
            19: 40e3,
            20: 40e3,
            21: 40e3,
            22: 50e3,
            23: 200e3,
            24: 200e3,
            25: 25e3,
            26: 25e3,
            27: 20e3,
            28: 70e3,
            29: 600e3,
            30: 70e3,
            31: 100e3,
            32: 40e3}.items()}

# Convert loads to per unit, one value per time step (nominal load at every hour)
load_p = {bus: [P / S_base for _ in range(N)] for bus, P in load_p_nominal.items()}
load_q = {bus: [Q / S_base for _ in range(N)] for bus, Q in load_q_nominal.items()}

## OPF Model

In [5]:
class BranchFlow_33bus():

    def __init__(self):
        self.m = ConcreteModel()
        self.define_variables()
        self.define_rootbus_constraints()
        self.define_voltagebalance_constraints()
        self.define_powerbalance_constraints()
        self.define_branchflow_constraint()
        self.define_objective_function()
        self.define_voltagelimits_constraints()
        self.solver = SolverFactory('ipopt')

    def define_variables(self):
        self.m.time = Set(initialize=t)
        self.m.generators = Set(initialize=[0])
        self.m.v = Var(bus_vars, self.m.time, domain=NonNegativeReals, initialize=Vmag)
        self.m.pline = Var(flow_vars, self.m.time, domain=Reals, initialize=0)
        self.m.qline = Var(flow_vars, self.m.time, domain=Reals, initialize=0)
        self.m.Iline = Var(flow_vars, self.m.time, domain=NonNegativeReals, initialize=0)
        self.m.Pg = Var(self.m.generators, self.m.time, domain=Reals, initialize=0)
        self.m.Qg = Var(self.m.generators, self.m.time, domain=Reals, initialize=0)

    def define_rootbus_constraints(self):
        self.m.rootbus_cons = ConstraintList()
        for hour in self.m.time:
            self.m.rootbus_cons.add(expr=self.m.v[0, hour] == Vmag)

    def define_objective_function(self):
        self.m.obj = Objective(
            expr=quicksum((self.m.Iline[fb, tb, hour]/I_base)**2 * np.real(Z[fb, tb]) for (fb, tb) in flow_vars for hour in self.m.time),
            sense=minimize)

    def define_branchflow_constraint(self):

        self.m.branchflow_cons = ConstraintList()
        for hour in self.m.time:
            for (fb, tb) in flow_vars:
                self.m.branchflow_cons.add(
                    expr=(self.m.pline[fb, tb, hour]/S_base)**2 + (self.m.qline[fb, tb, hour]/S_base)**2 ==
                         (self.m.v[fb, hour])**2 * (self.m.Iline[fb, tb, hour]/I_base)**2)

    def define_voltagebalance_constraints(self):
        self.m.voltagebalance_cons = ConstraintList()
        for hour in self.m.time:
            for (fb, tb) in flow_vars:
                self.m.voltagebalance_cons.add(expr=(self.m.v[tb, hour])**2 == (self.m.v[fb, hour])**2 -
                                               (2 * (self.m.pline[fb, tb, hour]/S_base * np.real(Z[fb, tb]) + self.m.qline[fb, tb, hour]/S_base * np.imag(Z[fb, tb])))
                                               + ((self.m.Iline[fb, tb, hour]/I_base)**2)*((np.real(Z[fb, tb]))**2 + (np.imag(Z[fb, tb]))**2))

    def define_voltagelimits_constraints(self):
        self.m.voltagelimits_cons = ConstraintList()
        for hour in self.m.time:
            for bus in bus_vars:
                self.m.voltagelimits_cons.add(expr=self.m.v[bus, hour] <= V_max)
                self.m.voltagelimits_cons.add(expr=self.m.v[bus, hour] >= V_min)

    def define_powerbalance_constraints(self):
        self.m.powerbalance_cons = ConstraintList()
        for hour in self.m.time:
            for bus in bus2flow:
                if bus == 0:
                    Pg = self.m.Pg[0, hour]/S_base
                    Qg = self.m.Qg[0, hour]/S_base
                else:
                    Pg = 0
                    Qg = 0
                p_out_flow = sum(self.m.pline[fb, tb, hour]/S_base for (fb, tb) in bus2flow[bus] if bus == fb)
                p_in_flow = sum(self.m.pline[fb, tb, hour]/S_base - (np.real(Z[fb, tb]) * (self.m.Iline[fb, tb, hour]/I_base)**2) for (fb, tb) in bus2flow[bus] if bus == tb)
                q_out_flow = sum(self.m.qline[fb, tb, hour]/S_base for (fb, tb) in bus2flow[bus] if bus == fb)
                q_in_flow = sum(self.m.qline[fb, tb, hour]/S_base - (np.imag(Z[fb, tb]) * (self.m.Iline[fb, tb, hour]/I_base)**2) for (fb, tb) in bus2flow[bus] if bus == tb)

                self.m.powerbalance_cons.add(expr=p_in_flow + (Pg - load_p[bus][hour]) - p_out_flow == 0)
                self.m.powerbalance_cons.add(expr=q_in_flow + (Qg - load_q[bus][hour]) - q_out_flow == 0)

    def solve(self):
        self.info = self.solver.solve(self.m)
        return self.info

## Solve

In [6]:
# solve the model
pfm = BranchFlow_33bus()
pfm.solve()

{'Problem': [{'Lower bound': -inf, 'Upper bound': inf, 'Number of objectives': 1, 'Number of constraints': 4728, 'Number of variables': 3144, 'Sense': 'unknown'}], 'Solver': [{'Status': 'ok', 'Message': 'Ipopt 3.14.20\\x3a Optimal Solution Found', 'Termination condition': 'optimal', 'Id': 0, 'Error rc': 0, 'Time': 0.22128605842590332}], 'Solution': [OrderedDict({'number of solutions': 0, 'number of solutions displayed': 0})]}

## Store Results

In [7]:
# Dictionaries to store results
V_buses = {}
P_flows_dict = {}
Q_flows_dict = {}
I_flows_dict = {}
branches_dict = {}
branches = [f"{fb}-{tb}" for (fb, tb) in flow_vars]
Pg_values=[]
losses_values=[]

# Extract and store results for each hour
for hour in t:
    # Bus voltages
    V_buses[hour] = [pfm.m.v[bus, hour]() for bus in bus_vars]
    # Branch flows
    P_flows_dict[hour] = [pfm.m.pline[fb, tb, hour]() for (fb, tb) in flow_vars]
    Q_flows_dict[hour] = [pfm.m.qline[fb, tb, hour]() for (fb, tb) in flow_vars]
    I_flows_dict[hour] = [pfm.m.Iline[fb, tb, hour]() for (fb, tb) in flow_vars]
    Pg_values.append(pfm.m.Pg[0, hour]()/1000)# To make it n KW
    losses_values.append(float((S_base/1000)*sum((pfm.m.Iline[fb, tb, hour]()/I_base)**2 * np.real(Z[fb, tb]) for (fb, tb) in flow_vars))) # in kW
# Store in a combined dictionary
    branches_dict[hour] = {
        branch: {
            "P": P_flows_dict[hour][i],
            "Q": Q_flows_dict[hour][i],
            "I": I_flows_dict[hour][i]
        } for i, branch in enumerate(branches)
    }
    

## Save Results

In [8]:
# === Create output folder ===
output_folder = f'Results/Obj(MinLosses)/Results_{Vmag}pu'
os.makedirs( output_folder, exist_ok=True)

# Convert voltage results to DataFrame
df_voltages = pd.DataFrame(V_buses, index=bus_vars)
df_voltages.index.name = 'Bus'
# sort columns (hours) numerically
df_voltages = df_voltages[sorted(df_voltages.columns)]
# Build file name
filename = os.path.join(
    output_folder,
    f"Voltage_Profiles_Data.csv"
)
# Save to CSV
df_voltages.to_csv(filename)

# save pg
with open(os.path.join(output_folder, 'Pg_Data.pkl'), "wb") as f:
    pickle.dump(Pg_values, f)
# save losses
with open(os.path.join(output_folder, 'Total_losses_Data.pkl'), "wb") as f:
    pickle.dump(losses_values, f)
# save branch flows
with open(os.path.join(output_folder, 'Branches_Data.pkl'), "wb") as f:
    pickle.dump(branches_dict, f)
print(f"Results saved to '{output_folder}'")

Results saved to 'Results/Obj(MinLosses)/Results_1.0pu'


## Voltage Profiles

In [9]:
######## PLOT VOLTAGE PROFILEs ########
# Set font to Times-like
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']


V_list = [V_buses[h] for h in range(N)]

timesteps = 1440  # 1-min resolution
if N == 1:
    time_range = np.linspace(0, 1, timesteps)
else:
    time_range = np.linspace(0, N-1, timesteps)

# Interpolate each bus (cubic needs at least 4 points)
interp_kind = 'cubic' if N >= 4 else 'linear'
V_buses_interp = [[0]*33 for _ in range(timesteps)]
for bus in range(33):
    hourly_voltage = [V_list[h][bus] for h in range(N)]
    if N == 1:
        interpolated = np.full(timesteps, hourly_voltage[0])
    else:
        f = interp1d(np.arange(N), hourly_voltage, kind=interp_kind)
        interpolated = f(time_range)
    for ts in range(timesteps):
        V_buses_interp[ts][bus] = interpolated[ts]

# Identify max & min bus per timestep
max_buses = [np.argmax(V_buses_interp[ts]) for ts in range(timesteps)]
min_buses = [np.argmin(V_buses_interp[ts]) for ts in range(timesteps)]

# --- Main Plot ---
fig, ax = plt.subplots(figsize=(12, 6))
plt.tick_params(axis='both', labelsize=18)

# Plot all buses in gray
for bus in range(33):
    ax.plot(time_range, [V_buses_interp[ts][bus] for ts in range(timesteps)], 
                color='lightgray', linewidth=0.8, alpha=0.5)

# Plot segmented max (red)
prev_bus = max_buses[0]
x_vals = [time_range[0]]
y_vals = [V_buses_interp[0][prev_bus]]
for ts in range(1, timesteps):
    current_bus = max_buses[ts]
    if current_bus == prev_bus:
        x_vals.append(time_range[ts])
        y_vals.append(V_buses_interp[ts][current_bus])
    else:
        ax.plot(x_vals, y_vals, color='red')
        prev_bus = current_bus
        x_vals = [time_range[ts]]
        y_vals = [V_buses_interp[ts][current_bus]]
if x_vals:
    ax.plot(x_vals, y_vals, color='red')

# Plot segmented min (blue)
prev_bus = min_buses[0]
x_vals = [time_range[0]]
y_vals = [V_buses_interp[0][prev_bus]]
for ts in range(1, timesteps):
    current_bus = min_buses[ts]
    if current_bus == prev_bus:
        x_vals.append(time_range[ts])
        y_vals.append(V_buses_interp[ts][current_bus])
    else:
        ax.plot(x_vals, y_vals, color='blue')
        prev_bus = current_bus
        x_vals = [time_range[ts]]
        y_vals = [V_buses_interp[ts][current_bus]]
if x_vals:
    ax.plot(x_vals, y_vals, color='blue')

# Format main plot
    
ax.set_xlabel('Hour of the Day', fontsize=18)
ax.set_ylabel('Voltage (p.u.)', fontsize=18)
ax.set_xticks(np.arange(0, max(N, 1) + 1, 2 if N >= 12 else 1))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
# --- Save or Show ---
plt.tight_layout()
plt.savefig(os.path.join(output_folder, 'Voltage_Profiles.pdf'), dpi=300, bbox_inches='tight')
plt.show()

/var/folders/x1/f1x_t6z1587654zd3d9kwcp80000gn/T/ipykernel_81956/2469780287.py:86: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
